# Dataset generators (offline, run-once)

Those generators have been saved in `iqp/data/*.json`. 

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import random
import itertools
from itertools import combinations

## MaxCut instance generator

In [ ]:
def random_maxcut_instance(n_qubits, p=0.5, weighted=True, seed=None):
    rng = np.random.default_rng(seed)
    edges, weights = [], []
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            if rng.random() < p:
                edges.append((i, j))
                weights.append(float(rng.integers(1, 11)) if weighted else 1.0)
    return edges, weights

## Number Partition instance generator (dung nghiem duy nhat)

In [ ]:
def count_partitions(nums):
    total = sum(nums)
    if total % 2 != 0:
        return 0
    target = total // 2
    n = len(nums)
    solutions = []
    for r in range(1, n):
        for subset in combinations(range(n), r):
            if sum(nums[i] for i in subset) == target:
                other = tuple(sorted(set(range(n)) - set(subset)))
                partition = tuple(sorted([tuple(sorted(subset)), other]))
                if partition not in solutions:
                    solutions.append(partition)
    return len(solutions)

def generate_unique_partition_instance(n=8, min_value=1, max_value=30, max_attempts=100000):
    for _ in range(max_attempts):
        nums = [random.randint(min_value, max_value) for _ in range(n)]
        if count_partitions(nums) == 1:
            return nums
    return None

## Ising exact ground state (brute-force) — sinh `ising_loss_exact.json`

In [ ]:
def exact_ising_solution(observables, coefficients):
    n_qubits = observables.shape[1]
    ground_energy = np.inf
    ground_state = None
    for spins in itertools.product([-1, 1], repeat=n_qubits):
        spins = np.array(spins)
        energy = 0.0
        for obs, coeff in zip(observables, coefficients):
            support = np.where(obs == 1)[0]
            value = np.prod(spins[support]) if len(support) else 1.0
            energy += coeff * value
        if energy < ground_energy:
            ground_energy = energy
            ground_state = spins.copy()
    return ground_state, ground_energy

```python
import json
qubits = [3, 6, 9, 12, 15]
data = {n: [random_maxcut_instance(n, seed=1000*n+j) for j in range(50)] for n in qubits}
json.dump({str(k): v for k, v in data.items()}, open('../data/maxcut.json', 'w'))
```